# Automatyzacja procesu księgowego

Celem projektu jest sprawdzenie możliwości automatyzacji procesu łączenia danych z systemu Allegro i systemu księgowego Optima. Obecnie nasza księgowa musi ręcznie przeszukiwać listę operacji z Allegro, wyszukiwać odpowiednie osoby w systemie Optima i przypisywać do nich numer dokumentu księgowego. Jest to proces żmudny i czasochłonny.

Wspólnie z CTO wpadliśmy na pomysł automatyzacji tego procesu, zaprezentuję ten koncept na bazie danych ze stycznia w poniższym notebooku.

INPUT:
- Raport Allegro
- Raport Optima

OUTPUT:
- Raport Allegro z dodanymi paragonami

In [ ]:
import numpy as np

In [ ]:
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#df_allegro_excel = pd.read_excel('2024-12-31_06-30_allegro.xlsx', parse_dates=['data'])
#df_optima_csv = pd.read_csv('01.01-20.07.2025.xls-Lista paragonów.tsv', sep='\t', parse_dates=['Data wyst.', 'Termin płatn.'])
path1 = '/content/drive/MyDrive/Allegro/2024-12-31_2025-06-30_794156668_pmnt_all.xlsx'
path2 = '/content/drive/MyDrive/Allegro/01.01-20.07.2025-optima.csv'

df_allegro_excel = pd.read_excel(path1, parse_dates=['data'])
df_optima_csv = pd.read_csv(path2, parse_dates=['Data wyst.'])

df_allegro = df_allegro_excel.copy()
df_optima = df_optima_csv.copy()

#### Dane z Optimy:

In [ ]:
df_optima.tail(20)

In [ ]:
df_optima.info()

Duplikaty

In [ ]:
df_optima['Odbiorca'].duplicated().sum()

#### Dane z Allegro:

In [ ]:
df_allegro.head()

In [ ]:
df_allegro.info()

Duplikaty:

In [ ]:
df_allegro['kupujący'].duplicated().sum()

#### Dane wymagają preprocessingu
1. df_allegro:
- ma 2 puste kolumny.
- ma wszystkie dane na temat kupującego w jednej kolumnie (Nickame;Imię Nazwiskoa;Ulica numer domu;Kod pocztowy Miasto).
- ma imię i nazwisko w różnych formach (raz z dużej litery a raz z małej).
- dane posiadają 38 duplikatów.
- kolumny z kwotą, saldem i dostawą są w typie object. Trzeba je zmienić na float dla późniejszego łączenia.

2. df_optima
- ma pusty wiersz na dole (podsumowanie).
- ma imię i nazwisko w różnych formach (raz z dużej litery a raz z małej).
- nieznaczne braki w 3 kolumnach.
- dane posiadają 70 duplikatów.
- kolumny Netto	Brutto są w typie object. Trzeba je zmienić na float dla późniejszego łączenia.

#### Dodatkowe wnioski:
Dodatkowo ilość wierszy nie jest zgodna (1175 allegro vs 1783 optima).
Dane z optimy są źle posortowane, a z allegro posortowane odwrotnie, dlatego będzie trzeba to ujednolicić.

## Data Preprocessing:

Usuwanie pustych kolumn i wierszy

In [ ]:
df_allegro.drop(['data zaksięgowania', 'szczegóły operacji'], axis=1, inplace=True)
df_optima.dropna(axis=0, inplace=True)

Potrzebuję wyłuskać imię i nazwisko z kolumny allegro

In [ ]:
df_allegro['Odbiorca'] = df_allegro['kupujący'].str.split(';').str[1]
df_allegro['Nickname'] = df_allegro['kupujący'].str.split(';').str[0]

Zauważyłem, że imię i nazwisko po którym będę łączył są wpisane raz dużymi literami, a raz małymi, więc muszę to ustandaryzować

In [ ]:
df_allegro['Odbiorca'] = df_allegro['Odbiorca'].str.strip().str.title()
df_optima['Odbiorca'] = df_optima['Odbiorca'].str.strip().str.title()

Zmiana typu danych w kolumnach z Object na Float

In [ ]:
for col in ['dostawa', 'kwota', 'saldo']:
    df_allegro[col] = df_allegro[col].str.replace(" zł", "", regex=False).str.replace(",", ".").astype(float)

In [ ]:
for col in ['Netto', 'Brutto']:
    df_optima[col] = df_optima[col].str.replace(",", ".").astype(float)

Sortowanie danych, od najwcześniejszych do najstarszych

In [ ]:
df_allegro = df_allegro.sort_values(by='data', ascending=True)
df_optima = df_optima.sort_values(by='Data wyst.', ascending=True)

Data wystawienia (Optima), zgadza się z datą raportu allegro więc spróbujemy je połączyć po imieniu i nazwisku oraz dacie i sprawdzimy rezultat

#### Teraz wybierzemy tylko kolumny, których potrzebujemy

1. Dane z allegro: Odbiorca, Nickname, kwota, operator, data, identyfikator

In [ ]:
df_allegro_clear = df_allegro[['identyfikator', 'Odbiorca', 'Nickname', 'kwota', 'operator', 'data']]
df_allegro_clear['matching data'] = df_allegro_clear['data'].dt.date
df_allegro_clear['matching data'] = pd.to_datetime(df_allegro_clear['matching data'], errors='coerce')
df_allegro_clear.shape

2. Dane z optimy: Odbiorca, Numer dokumentu

In [ ]:
df_optima = df_optima[['Odbiorca', 'Numer dokumentu', 'Data wyst.', 'Brutto']]
df_optima_clear = df_optima.rename(columns={'Data wyst.': 'matching data', 'Brutto':'kwota'})
df_optima_clear.shape

In [ ]:
df_allegro_clear.info()

## Łączenie danych
Dane są jużo oczyszczone ale dalej nie można ich połączyć po kolumnie Odbiorca, bo mają duplikaty.
Sprawdźmy dla ilu wierszy zgadza się Odbiorca z datą i kwotą:

In [ ]:
def zlicz_pasujace_wiersze(df1: pd.DataFrame, df2: pd.DataFrame, kolumny_df1: list, kolumny_df2: list) -> int:

    polaczone_df = pd.merge(
        left=df1,
        right=df2,
        left_on=kolumny_df1,
        right_on=kolumny_df2,
        how='inner'
    )

    return polaczone_df #len(polaczone_df)

In [ ]:
kolumny_allegro = ['Odbiorca', 'matching data']
kolumny_optima = ['Odbiorca', 'matching data']

liczba_dopasowan = zlicz_pasujace_wiersze(df_allegro_clear, df_optima_clear, kolumny_allegro, kolumny_optima)
zgodne = len(liczba_dopasowan)
print(f"Liczba wierszy, w których zgadzają się obie kolumny: {zgodne}")

In [ ]:
liczba_duplikatow = liczba_dopasowan.duplicated(subset=['Odbiorca', 'matching data']).sum()
brakujące_paragony = liczba_dopasowan['Numer dokumentu'].isna().sum()
zduplikowani_odbiorcy = liczba_dopasowan['Odbiorca'].duplicated().sum()
print(f'ilość brakujących paragonów: {brakujące_paragony}')
print(f'ilość zduplikowanych odbiorców: {zduplikowani_odbiorcy}')
print(f"Liczba wierszy, gdzie wartości w obu kolumnach się powtarzają: {liczba_duplikatow}")

1. odbiorca i data:
- wierszy 546, duplikatów 0, duplikaty w obu kolumnach 0
2. odbiorca i kwota:
- wierszy 876, duplikaty odbiorców 40, duplikaty w obu kolumnach 24

Wyniki mnie absolutnie nie satysfakcjonują dlatego sprawdzę czy nie lepszym rozwiązaniaem będzie pozbycie się duplikatów i potraktowanie ich osobno

In [ ]:
df_final = pd.merge(df_allegro_clear, df_optima_clear, on='Odbiorca', how='left')
df_final

In [ ]:
brakujące_paragony = df_final['Numer dokumentu'].isna().sum()
zduplikowani_odbiorcy = df_final['Odbiorca'].duplicated().sum()
print(f'ilość brakujących paragonów: {brakujące_paragony}')
print(f'ilość zduplikowanych odbiorców: {zduplikowani_odbiorcy}')

In [ ]:
duplicates = df_final[df_final.duplicated(subset='Odbiorca', keep=False)]
df_cleaned = df_final[~df_final.duplicated(subset='Odbiorca', keep=False)]
print(f'Tabela z duplikatami: {duplicates.shape}')
print(f'Czysta tabela: {df_cleaned.shape}')


In [ ]:
brakujące_paragony = df_cleaned['Numer dokumentu'].isna().sum()
zduplikowani_odbiorcy = df_cleaned['Odbiorca'].duplicated().sum()
print(f'ilość brakujących paragonów: {brakujące_paragony}')
print(f'ilość zduplikowanych odbiorców: {zduplikowani_odbiorcy}')

In [ ]:
df_cleaned.sample(10)

Sprawdźmy różnicę dni między dostarczeniem paragonu

In [ ]:
date_diff = (df_cleaned['matching data_x'] - df_cleaned['matching data_y']).dt.days.abs()
too_far_apart = df_cleaned[date_diff > 30].copy()
print(f'Znaleziono {too_far_apart.shape[0]} wierszy z różnicą dat > 30 dni')

In [ ]:
too_far_apart

Znaleziono błędy w Jest to niepokojące dlatego musimy przefiltować dodatkowo po cenie

In [ ]:
df_cleaned.shape

In [ ]:
df_broken = df_cleaned[(df_cleaned['kwota_x'] != df_cleaned['kwota_y'])]
df_broken.shape

In [ ]:
df_broken[df_broken['kwota_y'] > -0] #== None

In [ ]:
df_optima.loc[df_optima['Odbiorca'] == 'Natalia Płotka']

In [ ]:
# indeksy_do_usuniecia = df_cleaned[df_cleaned['Odbiorca'] == 'Natalia Szczotka'].index
# indeksy_do_usuniecia1 = df_cleaned[df_cleaned['Odbiorca'] == 'Kamil Gryczka'].index

# df_cleaned = df_cleaned.drop(indeksy_do_usuniecia)
# df_cleaned = df_cleaned.drop(indeksy_do_usuniecia1)

# df_cleaned.shape

- Po przefiltrowaniu danych jestem w stanie potwierdzić 1091/1175 rekordów (92%), ich odbiorca oraz kwota jest taka sama albo jest brak paragonu w optimie, są to niezduplikowane dane czyli nie ma miejsca na błąd.
- Zostały znalezione błędy (sprzeczene dane) w dwóch rekordach:
63 xxx oraz 19 xxx. Zaznaczę to na końcu dla księgowej na czerwono
- Zostały 86 niepotwierdzone rekordy.
- Zostało nam jeszcze 156 duplikatów, więc spróbujemy je wykorzystać żeby zminimalizować tę liczbę.

In [ ]:
true_duplicates = duplicates[duplicates['Numer dokumentu'].isna()]
true_duplicates.shape

In [ ]:
duplicates = duplicates[~duplicates['Numer dokumentu'].isna()]
duplicates.shape

Po usunięciu pustych danych zostaje nam 140 duplikatów na 70 pustych rekordów

In [ ]:
duplicates.sort_values(by='Odbiorca', ascending=True).head(15)

Ilość zduplikowanych transakcji się zwiększyła, świadczy to o tym że funkcja merge do każdej trnasakcji z df_allegro przypisała wszystkie paragony znalezione w df_optima

In [ ]:
df_allegro.loc[df_allegro['Odbiorca'] == 'Agata Kania']

In [ ]:
df_optima.loc[df_optima['Odbiorca'] == 'Krzysztof Rodak']

In [ ]:
duplicates.loc[duplicates['Odbiorca'] == 'Agata Kania']

Teraz widzimy jasno że dla każdego duplikatu funkcja merge przypisała wszystkie transakcje z takim samym imieniem

Po czym połączyć??
- Imię jest kluczowe ale daje nam 9 transakcji na 1
- Kwota może być pomocna ale co jeżeli osoba zamawia to samo
- Data wydaje się najlepszym warunkiem mimo tego że paragony często przychodzą później

Spróbujmy połączyć tylko te z taką samą datą, dodajmy kwotę jako dodatkowy filtr

In [ ]:
duplicates_data_mached = duplicates[duplicates['matching data_x'] == duplicates['matching data_y']]
duplicated_cleaned = duplicates_data_mached[duplicates_data_mached['kwota_x'] == duplicates_data_mached['kwota_y']]
duplicated_cleaned.shape

Udało się znaleźć 38 duplikatów z tą samą kwotą i datą, połączmy teraz wszystkie tabele:
- df_cleaned 1090
- true duplicates 16
- duplicated cleaned 38
1145/1175, resztę trzeba sprawdzić ręcznie

In [ ]:
df_definitive = pd.concat([df_cleaned, true_duplicates, duplicated_cleaned])
df_definitive.shape

Teraz wypełnię braki

In [ ]:
df_definitive[['Numer dokumentu',	',matching data_y',	'kwota_y']] = df_definitive[['Numer dokumentu',	'matching data_y',	'kwota_y']].fillna("brak w optimie")

Na koniec sprawdzę czy wszystko się zgadza

In [ ]:
df_filtered = df_allegro[~df_allegro['identyfikator'].isin(df_definitive['identyfikator'])]
df_filtered.shape

Tak jak widzimy w tabeli Allegro zostało 30 niezidentyfikowanych rekordów, wiemy że są to duplikaty, w których data albo kwota z optimą się nie zgadza

## Podsumowanie:
Po przeczyszczeniu, starannej analizie i połączeniu udało się potwierdzić numer paragonu albo jego brak dla 1145 wierszy na 1175, żeby rozjaśnić oraz uwiarygodnić moje działania wytłumaczę krok po kroku co zrobiłem:
1. połączyłem tabele po imieniu i nazwisku tak żeby do każdego imienia przypisało wszystkie możliwe transakcje z optimy z tym samym imieniem
2. odfiltrowałem tabelę z duplikatami tak żeby zostały same pojedyńcze potwierdzone wiersze, sprawdziłem w nich rozbieżność dat i tylko 4 wiersze miałwy większą rozbieżność niż 30 dni, zaniepokoiła mnie rozbieżność kwot więc przefiltrowałem całą tabelę i okazało się że w dwóch przypadkach sie nie zgadzają, zlokalizowałm je, usunąłem i czekają na anlizę
3. sprawdziłem duplikaty i odfiltrowałem 16 kontaktów, bo mogę potwierdzić że nie mają paragonów
4. resztę duplikatów przefiltowałem po dacie i kwocie żeby móc potwierdzić ich słszność
5. tabele połączyłem w całość
6. braki wypełniłem tekstem "brak w optimie", będzie to istotne z punktu widzenia księgowej


In [ ]:
#df_definitive.to_excel('/content/drive/MyDrive/Allegro/definitive_data.xlsx', index=False)

# Kolejnym krokiem będzie wczytanie całej tabeli i podłączenie danych

In [ ]:
df_full = pd.read_excel('/content/drive/MyDrive/Allegro/2024-12-31_2025-06-30_operations_all.xlsx')

In [ ]:
df_full

Na początku pozbędę się niepotrzebnych kolumn

In [ ]:
df_full.drop(['data zaksięgowania', 'dostawa', 'szczegóły operacji', 'saldo'], axis=1, inplace=True)

Teraz pozbędziemy się niepotrzebnych kolumn w naszej definitywnej tabeli

In [ ]:
df_definitive.info()

In [ ]:
df_definitive_to_merge = df_definitive[['identyfikator', 'Numer dokumentu']]

In [ ]:
gotowa_tabela = pd.merge(df_full, df_definitive_to_merge, on='identyfikator', how='left')

In [ ]:
braki = gotowa_tabela['Numer dokumentu'].isna().sum()
2413 - braki

In [ ]:
zwroty = df_full.loc[df_full['operacja'] == 'zwrot']
zwroty.shape

widzimy że paragon został przypisany również do zwrotów

#### Podsumowanie: Zestawienie operacji Allegro 1.01.2025-30.06.2025
Mamy 2413 wiersze.
W 1744 wierszach zostały, powierdzone paragony lub ich brak w optimie
na składają sie na to:
- wiersze z takim samym imieniem i nazwiskiem z optimy, podobną datą oraz taką samą kwotą lub jej brakiem
- duplikaty z brakiem paragonu
- duplikaty ale podzielone na te z taką samą datą oraz kwotą
- zwroty z takim samym paragonem jak zamówienia
- znaleziono dwa podejrzane rekordy, zostały zaznaczone w tabeli


In [ ]:
gotowa_tabela.to_excel('/content/drive/MyDrive/Allegro/gotowa_tabela.xlsx', index=False)